# argmax-accuracy-eval — ex2: weighted accuracy accumulator across a val loader (partial last batch)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `argmax-accuracy-eval`. Running the final beacon cell reports progress against the `Eval: argmax accuracy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Eval: argmax accuracy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`argmax-accuracy-eval`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "argmax-accuracy-eval"
DD_SUBTOPIC = "Eval: argmax accuracy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Weighted-average accuracy across a val loader — quick refresher

A single-batch accuracy is `(logits.argmax(-1) == y).float().mean()`. Over a FULL val loader you can't just average per-batch accuracies naively: the last batch is often partial, and the unweighted mean over-counts it. The correct accumulator weights each batch by its size:

```python
correct, total = 0, 0
with t.inference_mode():
    for x, y in loader:
        preds   = model(x).argmax(dim=-1)
        correct += (preds == y).sum().item()
        total   += y.shape[0]
acc = correct / total          # weighted by per-batch size
```

Numerator counts integer hits; denominator counts examples. The ratio is mathematically identical to `mean of per-example correct` over the WHOLE eval set, regardless of how the loader batched it.

### Exercise 2 — weighted accuracy accumulator across a val loader (partial last batch)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a `(correct, total)` integer accumulator over a val loader so the reported accuracy is exactly correct even when the last batch is partial.
> Keywords: accuracy, val-loader, partial-batch, weighted-average
> ```

**KCs targeted:** `accuracy-weighted-by-batch-size`, `argmax-along-class-dim-minus-1`

Implement `ex2_eval_accuracy(model, loader)`. Run the model over the whole loader and return overall top-1 accuracy as a Python float in `[0, 1]`.

1. Initialize integer counters `correct = 0`, `total = 0`.
2. Under `t.inference_mode():`, iterate `(x, y) in loader`:
   - `preds = model(x).argmax(dim=-1)` (shape matches `y`).
   - `correct += (preds == y).sum().item()` — accumulate as int.
   - `total   += y.shape[0]` — accumulate the per-batch size.
3. Return `correct / total` (Python float).

The test passes a loader whose LAST batch is partial (size 3 vs batch_size 8) and verifies the result matches the exact ratio computed by hand on the full dataset. A wrong implementation that averages per-batch accuracies fails this test because it weights the small last batch the same as the full ones.

In [ ]:
def ex2_eval_accuracy(model, loader) -> float:
    """Weighted top-1 accuracy across a loader; handles partial last batch."""
    raise NotImplementedError()


def _test_ex2():
    from torch.utils.data import TensorDataset, DataLoader

    # === Construct a dataset where we KNOW the per-example correctness ===
    # 19 examples, batch_size=8 → batches of sizes [8, 8, 3] (partial last).
    B, C = 19, 4
    rng = t.Generator().manual_seed(0)
    X = t.randn(B, 6, generator=rng)
    Y = t.randint(0, C, (B,), generator=rng)
    ds = TensorDataset(X, Y)
    loader = DataLoader(ds, batch_size=8, shuffle=False)
    # Verify the loader actually emits a partial last batch.
    batch_sizes = [yy.shape[0] for _, yy in loader]
    assert batch_sizes == [8, 8, 3], f'unexpected batch sizes {batch_sizes}'

    model = t.nn.Linear(6, C)
    model.eval()

    # Ground truth: compute hits over the entire dataset directly.
    with t.inference_mode():
        full_preds = model(X).argmax(dim=-1)
    ref_correct = int((full_preds == Y).sum().item())
    ref_acc = ref_correct / B

    acc = ex2_eval_accuracy(model, loader)
    assert isinstance(acc, float), f'must return float, got {type(acc).__name__}'
    assert 0.0 <= acc <= 1.0, f'accuracy out of [0,1]: {acc}'
    assert abs(acc - ref_acc) < 1e-6, (
        f'weighted acc {acc} != reference {ref_acc}; partial-last-batch handling wrong'
    )

    # === Sanity: perfect-classifier case ===
    class _Perfect(t.nn.Module):
        def __init__(self, n_classes):
            super().__init__()
            self.n_classes = n_classes
        def forward(self, x):
            # Use first column of x as the 'label hint' and produce confident logits.
            labels = self._latest_labels
            out = t.full((x.shape[0], self.n_classes), -10.0)
            out[t.arange(x.shape[0]), labels] = 10.0
            return out

    # Build a loader where the model has access to the labels (cheat-perfect).
    class _LabelLeakLoader:
        def __init__(self, X, Y, batch_size):
            self.X = X
            self.Y = Y
            self.batch_size = batch_size
        def __iter__(self):
            for i in range(0, len(self.X), self.batch_size):
                xb = self.X[i:i+self.batch_size]
                yb = self.Y[i:i+self.batch_size]
                self._cur_y = yb
                yield xb, yb

    perfect = _Perfect(C)
    leak = _LabelLeakLoader(X, Y, batch_size=8)
    # Hook labels into model right before forward by patching iterator.
    class _CheatModel(t.nn.Module):
        def __init__(self, n_classes, X, Y):
            super().__init__()
            self.n_classes = n_classes
            self.X = X
            self.Y = Y
        def forward(self, x):
            # Match each row of x to its index in self.X to find its label.
            out = t.full((x.shape[0], self.n_classes), -10.0)
            for i, row in enumerate(x):
                j = ((self.X - row).abs().sum(dim=-1) < 1e-6).nonzero(as_tuple=True)[0][0]
                out[i, self.Y[j]] = 10.0
            return out

    cheat = _CheatModel(C, X, Y)
    perfect_acc = ex2_eval_accuracy(cheat, loader)
    assert perfect_acc == 1.0, f'cheating-perfect model must give acc=1.0; got {perfect_acc}'

    # === Sanity: empty loader → would div-by-zero; test small but non-empty ===
    tiny_loader = DataLoader(TensorDataset(X[:1], Y[:1]), batch_size=1)
    tiny_acc = ex2_eval_accuracy(model, tiny_loader)
    assert tiny_acc in (0.0, 1.0), f'single-example loader gives 0 or 1; got {tiny_acc}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_eval_accuracy(model, loader):
    correct = 0
    total = 0
    with t.inference_mode():
        for x, y in loader:
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.shape[0]
    return correct / total
```

**Why integer accumulators, not float means.** `(correct / total)` is exact arithmetic over hit counts. A naive average-of-batch-means is biased whenever batches differ in size — the partial last batch is the canonical case. Same trick applies to weighted-average val loss (multiply per-batch loss by `x.shape[0]`).

**Why `.item()` inside the loop.** Without it, you'd hold a growing list of 0-D tensors in autograd memory across the whole loader. `inference_mode` blocks autograd already, but `.item()` also detaches and releases the scalar — saves memory on large eval sets.

**Top-K extension.** Replace `argmax(dim=-1)` with `topk(K, dim=-1).indices`, then `correct += (labels.unsqueeze(-1) == topk_idx).any(dim=-1).sum().item()`. Same accumulator pattern.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()